# 01 · Extract FaceForensics++ face crops (training data + in-dataset test set)

Turns FF++ videos into face-crop frames in `train/val/test` × `real/fake`, split by identity so no face appears in two splits.

**Notebook settings (right sidebar)**
- **Accelerator:** None (CPU is enough)
- **Internet:** On
- **Inputs:** a FaceForensics++ **video** dataset; the project code (optional, otherwise cloned)

**Produces:** `/kaggle/working/data` (face crops + `manifest.csv`). When it finishes, use **Save Version → Save & Run All**, then add this notebook's output as an input to the next notebook.

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
from src.kaggle_utils import setup
setup(data_dir="/kaggle/working/data")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

## Settings
`MAX_VIDEOS` limits videos per method; `None` uses everything (slowest). `FRAMES` is frames per fake video; real videos get proportionally more so classes balance.

In [ ]:
MAX_VIDEOS = 400      # None = all videos
FRAMES = 10
COMPRESSION = "c23"   # raw / c23 / c40, ignored if the dataset has only one
SPLITS_DIR = None     # folder with official train.json/val.json/test.json, if you have it

vids = [v for v in glob.glob("/kaggle/input/**/*.mp4", recursive=True)]
assert vids, "No .mp4 files found: add a FaceForensics++ video dataset as an input"
FF_SRC = os.path.commonpath([os.path.dirname(v) for v in vids])
print(f"FF++ root: {FF_SRC}  ({len(vids)} videos)")

In [ ]:
args = f'--src "{FF_SRC}" --frames {FRAMES} --compression {COMPRESSION} --workers 4'
if MAX_VIDEOS: args += f" --max-videos {MAX_VIDEOS}"
if SPLITS_DIR: args += f' --splits "{SPLITS_DIR}"'
!python -m src.classifier.ff_extract {args}

## What was extracted

In [ ]:
m = pd.read_csv("/kaggle/working/data/manifest.csv")
display(m.pivot_table(index="method", columns="split", values="file", aggfunc="count", fill_value=0))
display(m.groupby(["split", "label"]).video.nunique().unstack().rename_axis(None).style.set_caption("videos per split"))

In [ ]:
methods = sorted(m.method.unique())
fig, axes = plt.subplots(len(methods), 6, figsize=(14, 2.4 * len(methods)))
for row, method in zip(np.atleast_2d(axes), methods):
    sample = m[(m.method == method) & (m.split == "train")].sample(6, random_state=0)
    for ax, r in zip(row, sample.itertuples()):
        ax.imshow(Image.open(f"/kaggle/working/data/{r.split}/{r.label}/{r.file}"))
        ax.set_title(f"{method}\n{r.video}", fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Keep only the data in this notebook's output
shutil.rmtree("/kaggle/working/outputs", ignore_errors=True)
!du -sh /kaggle/working/data